# Exercise 7 — Trajectory Visualiser

Run this notebook after completing the experiment in `template.py` (or `maze_practice_solution.py`).

Each run of the experiment creates a timestamped folder (`run_YYYYMMDD_HHMMSS/`) containing:
- `maze_practice.csv` — one row per trial (condition, stars, duration)
- `trajectory.csv` — player position sampled every 0.1 s
- `maze_walls.csv` — wall geometry for each trial

This notebook automatically loads the **most recent** run folder.

It produces:
1. **Top-down maze map** per trial — walls, path, start/end, collection events
2. **Position over time** plot — x and z traces with collection markers

In [ ]:
import csv
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Find the most recent run_* folder next to this notebook (or the notebook's parent)
_base_dirs = [Path(os.path.abspath('')), Path(__file__).parent if '__file__' in dir() else Path('.')]
DATA_DIR = None
for _base in _base_dirs:
    _runs = sorted(_base.glob('run_*'), reverse=True)
    if _runs:
        DATA_DIR = _runs[0]
        break

if DATA_DIR is None:
    raise RuntimeError(
        'No run_* folder found. Run template.py or maze_practice_solution.py first, '
        'then re-run this notebook.'
    )

print(f'Loading run: {DATA_DIR.name}  ({DATA_DIR.resolve()})')

In [ ]:
def load_csv(path):
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

exp_rows  = load_csv(DATA_DIR / 'maze_practice.csv')
traj_rows = load_csv(DATA_DIR / 'trajectory.csv')

# maze_walls.csv is written by template.py / solution.py
walls_path = DATA_DIR / 'maze_walls.csv'
wall_rows  = load_csv(walls_path) if walls_path.exists() else []
if not wall_rows:
    print('maze_walls.csv not found — maze outlines will be skipped.')

print(f'Trials:             {len(exp_rows)}')
print(f'Trajectory samples: {len(traj_rows)}')
print(f'Wall segments:      {len(wall_rows)}')

## Group data by trial

In [ ]:
traj_by_trial  = defaultdict(list)   # trial -> [(x, z), ...]
event_by_trial = defaultdict(list)   # trial -> [(x, z), ...]  (collect events)
wall_by_trial  = defaultdict(list)   # trial -> [(x, z, sx, sz), ...]
time_by_trial  = defaultdict(list)   # trial -> [(time_s, x, z), ...]

for r in traj_rows:
    trial = int(r['trial'])
    x, z  = float(r['x']), float(r['z'])
    traj_by_trial[trial].append((x, z))
    time_by_trial[trial].append((float(r['time_s']), x, z))
    if r['event'].startswith('collect'):
        event_by_trial[trial].append((x, z))

for r in wall_rows:
    trial = int(r['trial'])
    wall_by_trial[trial].append((
        float(r['x']), float(r['z']),
        float(r['sx']), float(r['sz']),
    ))

## Plot 1 — Top-down maze map

In [ ]:
n_trials = len(exp_rows)
if n_trials == 0:
    raise RuntimeError(
        'maze_practice.csv has no trial rows — run template.py first, then re-run this notebook.'
    )

ncols = 2
nrows = (n_trials + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 8 * nrows), squeeze=False)
axes_flat = axes.flatten()

for i, exp_row in enumerate(exp_rows):
    ax    = axes_flat[i]
    trial = int(exp_row['trial'])
    cond  = exp_row['condition']

    # Draw maze walls
    for (wx, wz, sx, sz) in wall_by_trial[trial]:
        rect = mpatches.Rectangle(
            (wx - sx / 2, wz - sz / 2), sx, sz,
            linewidth=0, facecolor='#555555', zorder=1,
        )
        ax.add_patch(rect)

    # Draw trajectory
    pts = traj_by_trial[trial]
    if pts:
        xs, zs = zip(*pts)
        ax.plot(xs, zs, color='royalblue', linewidth=1.5, zorder=2, label='path')
        ax.plot(xs[0],  zs[0],  'o', color='limegreen', markersize=10, zorder=4, label='start')
        ax.plot(xs[-1], zs[-1], 's', color='crimson',   markersize=8,  zorder=4, label='end')

    # Draw collection events
    for (ex, ez) in event_by_trial[trial]:
        ax.plot(ex, ez, '*', color='gold', markersize=16,
                markeredgecolor='darkorange', markeredgewidth=0.8,
                zorder=5, label='collect')

    ax.set_aspect('equal')
    ax.invert_yaxis()   # row 0 (small z) at top, matches game layout
    ax.set_facecolor('#1a1a2e')
    ax.set_title(
        f"Trial {trial}  [{cond}]   "
        f"{exp_row['collected']}/{exp_row['n_stars']} stars   "
        f"{float(exp_row['duration_s']):.1f} s",
        fontsize=12,
    )
    ax.set_xlabel('x (world units)')
    ax.set_ylabel('z (world units)')

    # Deduplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    seen = {}
    for h, l in zip(handles, labels):
        seen.setdefault(l, h)
    ax.legend(seen.values(), seen.keys(), loc='upper right', fontsize=9)

# Hide unused subplots
for j in range(n_trials, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Maze Explorer — Trajectory Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()

out = DATA_DIR / 'trajectory_plot.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved → {out.resolve()}')
plt.show()

## Plot 2 — Position over time

In [ ]:
fig2, axes2 = plt.subplots(n_trials, 1, figsize=(12, 3 * n_trials), squeeze=False)

for i, exp_row in enumerate(exp_rows):
    ax    = axes2[i][0]
    trial = int(exp_row['trial'])

    times, xs, zs = zip(*time_by_trial[trial]) if time_by_trial[trial] else ([], [], [])

    ax.plot(times, xs, label='x', color='steelblue')
    ax.plot(times, zs, label='z', color='tomato')

    # Mark collection events with vertical lines
    for r in traj_rows:
        if int(r['trial']) == trial and r['event'].startswith('collect'):
            ax.axvline(float(r['time_s']), color='gold', linestyle='--', linewidth=1.5)

    ax.set_title(
        f"Trial {trial}  [{exp_row['condition']}]   "
        f"{exp_row['collected']}/{exp_row['n_stars']} stars",
        fontsize=11,
    )
    ax.set_xlabel('time (s)')
    ax.set_ylabel('position')
    ax.legend(loc='upper right', fontsize=9)

fig2.suptitle('Position over Time  (gold dashes = star collected)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()